# Metrics for text, translation & speech

_Metrics & Evaluation — measuring models, data & statistics_

**How to put a number on a machine's words — comparing what it wrote to what a human wrote.**

How do you grade a sentence a machine wrote? You compare it to a reference — a sentence a human wrote that you trust. The metric turns "how close are these two sentences?" into a number.

       There are three big families, and the whole lesson hangs on this split:

---

This notebook is a practice scaffold for the **Metrics for text, translation & speech** lesson. We walk through one metric family at a time — overlap (BLEU/ROUGE), then meaning (METEOR/BERTScore), then edit-distance (WER/CER) — so you can see exactly what each number rewards. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — sacrebleu, rouge_score, evaluate, jiwer, torchmetrics

### Step 1 — Overlap metrics with sacrebleu (BLEU, chrF, TER)

The first family scores a candidate by how many word/character n-grams it shares with a human reference. `sacrebleu` is the standard, reproducible implementation. BLEU and chrF are higher-is-better (0–100); TER is an edit rate, so lower is better. The pip line installs the libraries this whole notebook needs.

In [ ]:
# pip install sacrebleu rouge-score evaluate jiwer torchmetrics torch
import sacrebleu

# sacrebleu expects a list of reference lists (one list of refs per sentence).
refs = [["the quick brown fox jumps over the lazy dog"]]
cand = ["a quick brown fox jumped over the lazy dog"]

bleu = sacrebleu.corpus_bleu(cand, refs)   # 0..100, higher better
chrf = sacrebleu.corpus_chrf(cand, refs)   # character F-score
ter  = sacrebleu.corpus_ter(cand, refs)    # edit rate, lower better
print("BLEU :", bleu.score)
print("chrF :", chrf.score)
print("TER  :", ter.score)

### Step 2 — Summarization overlap with ROUGE

ROUGE is the overlap metric of choice for summarization: it measures how much of the reference's content the candidate recovers. ROUGE-1 and ROUGE-2 count shared unigrams and bigrams, while ROUGE-L uses the longest common subsequence. Stemming lets word variants like "jumps"/"jumped" still match.

In [ ]:
from rouge_score import rouge_scorer

# use_stemmer=True so jumps/jumped count as a match.
scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

reference = "the quick brown fox jumps over the lazy dog"
candidate = "a quick brown fox jumped over the lazy dog"
rouge_scores = scorer.score(reference, candidate)
print(rouge_scores)

### Step 3 — Meaning-aware metrics via HuggingFace evaluate

Overlap metrics miss paraphrases, so the next family judges *meaning*. The `evaluate` hub loads many metrics behind one interface: METEOR (overlap with synonym/stem matching), BERTScore (embedding similarity, so semantically close words count), and SQuAD's exact-match + token-F1 for question answering.

In [ ]:
import evaluate

# evaluate wants flat lists of strings, so unwrap each reference list.
references = [r[0] for r in refs]

meteor = evaluate.load("meteor")
meteor_score = meteor.compute(predictions=cand, references=references)
print("METEOR:", meteor_score)

# BERTScore compares contextual embeddings, so it rewards meaning, not just words.
bertscore = evaluate.load("bertscore")
bert_result = bertscore.compute(predictions=cand, references=references, lang="en")
print("BERTScore:", bert_result["f1"])

# SQuAD metric: exact match + token-level F1 for QA answers.
squad = evaluate.load("squad")
squad_score = squad.compute(
    predictions=[{"id": "1", "prediction_text": "Paris"}],
    references=[{"id": "1", "answers": {"text": ["Paris"], "answer_start": [0]}}])
print(squad_score)

### Step 4 — Edit-distance metrics for speech/OCR, plus torchmetrics

The third family scores transcripts by counting edits. WER (Word Error Rate) and CER (Character Error Rate) from `jiwer` are the speech/OCR standards — lower is better. Finally, `torchmetrics.text` offers BLEU, ROUGE, BERTScore, and **perplexity** (exp of the mean cross-entropy) as drop-in PyTorch modules; perplexity needs per-token logits and the true next-token ids.

In [ ]:
import jiwer

# WER/CER count word/character edits; lower is better.
print("WER:", jiwer.wer("turn the lights on now", "turn lights on now"))   # 0.2
print("CER:", jiwer.cer("turn the lights on now", "turn lights on now"))

import torch
from torchmetrics.text import Perplexity, BLEUScore, ROUGEScore, BERTScore

ref_list = [r[0] for r in refs]
print("BLEU (tm):", BLEUScore()(cand, [ref_list]))
print("ROUGE (tm):", ROUGEScore()(cand, ref_list))

# Perplexity wants per-token logits (V = vocab size) and the true next-token ids:
logits = torch.randn(2, 8, 100)          # (batch, seq_len, vocab) model outputs
target = torch.randint(0, 100, (2, 8))   # (batch, seq_len) gold tokens
print("Perplexity:", Perplexity()(logits, target))   # exp of mean cross-entropy
print("BERTScore (tm):", BERTScore()(cand, ref_list)["f1"])

## Visualize the data & results

_Given one human reference and three machine candidates, do the metrics agree on which candidate is best — high BLEU and ROUGE, low WER for the good one?_

### Step 1 — Set up the reference and three candidates

To see whether the metrics agree, we fix one human reference and three machine candidates: an exact copy, a near-paraphrase with two changed words, and a mostly-different sentence. A good metric should rank A best and C worst. We also define a tokenizer so every metric splits sentences the same way.

In [ ]:
from collections import Counter

reference = "the quick brown fox jumps over the lazy dog"
candidates = {
    "A": "the quick brown fox jumps over the lazy dog",   # exact copy
    "B": "a quick brown fox jumped over the lazy dog",     # two word changes
    "C": "the dog ran past a brown fox",                   # mostly different
}

def tokenize(s):
    return s.split()

### Step 2 — Implement BLEU-1, ROUGE-1, and WER from scratch

Now build small versions of each metric so the numbers are transparent. **WER** is the Levenshtein edit distance (deletions + insertions + substitutions) divided by reference length. **BLEU-1** is clipped unigram *precision*, and **ROUGE-1** is the F1 of unigram precision and recall — both built from the same clipped-overlap count.

In [ ]:
def wer(ref, hyp):
    # Word Error Rate via the Levenshtein edit-distance table.
    r, h = tokenize(ref), tokenize(hyp)
    n, m = len(r), len(h)
    D = np.zeros((n + 1, m + 1), dtype=int)
    D[:, 0] = np.arange(n + 1)           # cost of deleting all reference words
    D[0, :] = np.arange(m + 1)           # cost of inserting all hypothesis words
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            cost = 0 if r[i - 1] == h[j - 1] else 1
            deletion = D[i - 1, j] + 1
            insertion = D[i, j - 1] + 1
            match_or_sub = D[i - 1, j - 1] + cost
            D[i, j] = min(deletion, insertion, match_or_sub)
    return D[n, m] / max(n, 1)

def clipped_overlap(ref, hyp):
    # Shared unigrams, each clipped to how often it appears in the reference.
    rc, hc = Counter(tokenize(ref)), Counter(tokenize(hyp))
    return sum(min(hc[w], rc[w]) for w in hc)

def bleu1(ref, hyp):
    # BLEU-1 = clipped unigram precision.
    overlap = clipped_overlap(ref, hyp)
    return overlap / max(len(tokenize(hyp)), 1)

def rouge1_f1(ref, hyp):
    # ROUGE-1 = F1 of unigram precision and recall.
    overlap = clipped_overlap(ref, hyp)
    rec = overlap / max(len(tokenize(ref)), 1)
    prec = overlap / max(len(tokenize(hyp)), 1)
    if rec + prec == 0:
        return 0.0
    return 2 * rec * prec / (rec + prec)

### Step 3 — Score the candidates and plot

Finally, run all three metrics on each candidate and chart them side by side. We expect BLEU-1 and ROUGE-1 (higher = better) to fall from A to C, while WER (lower = better) rises — confirming the metrics agree on which candidate is best.

In [ ]:
for name, cand_text in candidates.items():
    print(name,
          "BLEU-1=%.3f" % bleu1(reference, cand_text),
          "ROUGE-1=%.3f" % rouge1_f1(reference, cand_text),
          "WER=%.3f" % wer(reference, cand_text))
# A BLEU-1=1.000 ROUGE-1=1.000 WER=0.000
# B BLEU-1=0.778 ROUGE-1=0.778 WER=0.222
# C BLEU-1=0.571 ROUGE-1=0.500 WER=0.889

names = list(candidates)
x = np.arange(len(names))
bleu_vals = [bleu1(reference, candidates[n]) for n in names]
rouge_vals = [rouge1_f1(reference, candidates[n]) for n in names]
wer_vals = [wer(reference, candidates[n]) for n in names]

plt.bar(x - 0.25, bleu_vals, 0.25, label="BLEU-1", color="#4ea1ff")
plt.bar(x + 0.00, rouge_vals, 0.25, label="ROUGE-1", color="#7ee787")
plt.bar(x + 0.25, wer_vals, 0.25, label="WER", color="#ffb454")
plt.xticks(x, names)
plt.ylabel("score (0..1)")
plt.legend()
plt.title("BLEU-1 & ROUGE-1 up, WER down for the better candidate")
plt.show()

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Reference: cats sit on mats (4 words). Candidate: cats sit on the mat (5 words). Compute the unigram (1-gram) precision $p_1$ and ROUGE-1 recall.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- List candidate words: cats, sit, on, the, mat. Which appear in the reference {cats, sit, on, mats}? cats, sit, on do; "the" and "mat" do not. — _Precision counts candidate words found in the reference._
- $p_1 = 3/5 = 0.6$. — _3 of the 5 candidate words matched._
- Recall counts reference words produced: of {cats, sit, on, mats}, the candidate has cats, sit, on (not "mats", only "mat"). So recall $= 3/4 = 0.75$. — _"mat" $\ne$ "mats" — exact word match fails, which is the paraphrase blind spot._

**Answer:** Unigram precision $p_1 = 3/5 = 0.6$; ROUGE-1 recall $= 3/4 = 0.75$. Note "mat" vs "mats" costs a match even though the meaning is the same — the classic overlap-metric weakness.

</details>

**Problem 2.** A speech model transcribes the 5-word reference turn the lights on now as turn lights on now. What is the WER (Word Error Rate)?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Align the two: the only difference is the missing word "the". — _Find the cheapest set of single-word edits._
- That is one deletion: $S=0, D=1, I=0$. — _Deleting "the" from the reference gives the candidate._
- $\text{WER} = (S+D+I)/(\text{reference words}) = 1/5 = 0.2$. — _Divide edits by the 5 reference words to get a rate._

**Answer:** WER $= 1/5 = 0.2$ (one deleted word out of five). Lower is better, so this is a fairly good transcript.

</details>

**Problem 3.** Your translation model scores high BLEU but a reviewer says the outputs read like copies of the source. Which metric should you add, and why?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- BLEU and ROUGE reward shared n-grams, so copying scores well — they cannot see "is this novel or just pasted?". — _Overlap metrics are blind to copying and to paraphrase quality._
- Add a meaning-aware metric (BERTScore or COMET) to check semantic adequacy, and a diversity/novelty check (distinct-n or self-BLEU) to catch copying. — _Embedding metrics judge meaning; diversity metrics judge repetition._

**Answer:** Pair BLEU with an embedding-based metric (BERTScore / COMET) for meaning, and a diversity metric (distinct-n or self-BLEU) to detect copying. No single overlap number should be the only judge.

</details>